# 06b — Schema Prep From Existing Identity Results

**This notebook does not rerun SCRFD, AdaFace, embeddings, tracklets, or Person_ID clustering.**

It reads the files already produced by `06_top2_production_identity_pipeline(2).ipynb` and converts them into the schema-ready database/handoff format.

Expected existing files:

- `Results/video_inventory.csv`
- `Results/final_identity_observations.csv`
- `Results/final_identity_summary.csv` (optional; rebuilt if missing)

It preserves the existing `person_id` assignments exactly. The only new IDs created here are:

- integer `video_id`
- schema-ready `frame_id`
- deterministic integer `detection_id`


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# POINT THIS TO THE RUN YOU ALREADY COMPLETED
# ============================================================

RUN_MODE = "all"   # change to "test" only if your completed run was in test/

ROOT = Path(".")
RUN_ROOT = ROOT / "Data" / "IdentityPipelineTop2" / RUN_MODE

RESULTS_DIR = RUN_ROOT / "Results"
HANDOFF_DIR = RUN_ROOT / "Handoffs"
DATABASE_DIR = RUN_ROOT / "Database"

HANDOFF_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_INVENTORY_CSV = RESULTS_DIR / "video_inventory.csv"
FINAL_FACE_OBSERVATIONS_CSV = RESULTS_DIR / "final_identity_observations.csv"
FINAL_SUMMARY_CSV = RESULTS_DIR / "final_identity_summary.csv"

required = [
    VIDEO_INVENTORY_CSV,
    FINAL_FACE_OBSERVATIONS_CSV,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing existing pipeline outputs:\n- " + "\n- ".join(missing)
    )

print("Using completed run:", RUN_ROOT.resolve())
print("No models will be loaded or rerun.")


In [ ]:
# ============================================================
# LOAD THE ALREADY-COMPUTED IDENTITY RESULTS
# ============================================================

df_videos_old = pd.read_csv(VIDEO_INVENTORY_CSV)
df_final_faces = pd.read_csv(FINAL_FACE_OBSERVATIONS_CSV)

if FINAL_SUMMARY_CSV.exists():
    person_summary = pd.read_csv(FINAL_SUMMARY_CSV)
else:
    person_summary = (
        df_final_faces
        .groupby("person_id")
        .agg(
            occurrences=("frame_id", "nunique"),
            observations=("observation_id", "count"),
            videos=("source_video", "nunique"),
            mean_face_quality=("face_quality_score", "mean"),
            mean_frontal_score=("frontal_score", "mean"),
            mean_det_score=("det_score", "mean"),
        )
        .reset_index()
    )

print("Loaded face observations:", f"{len(df_final_faces):,}")
print("Existing Person_IDs:", f"{df_final_faces['person_id'].nunique():,}")
print("Existing videos:", f"{len(df_videos_old):,}")

# Important safety check: we are using the already-clustered Person_IDs.
assert "person_id" in df_final_faces.columns
assert df_final_faces["person_id"].notna().all()


In [ ]:
# ============================================================
# CREATE SCHEMA-READY VIDEO / FRAME / DETECTION KEYS
# WITHOUT RECLUSTERING ANYONE
# ============================================================

required_video_cols = {
    "video_id",
    "source_video",
    "source_path",
    "duration_seconds",
    "reported_fps",
    "width",
    "height",
}

required_face_cols = {
    "video_id",
    "source_video",
    "source_frame_index",
    "source_time_seconds",
    "person_id",
    "observation_id",
    "face_bbox_x1",
    "face_bbox_y1",
    "face_bbox_x2",
    "face_bbox_y2",
    "det_score",
    "selected_downstream_frame",
    "full_frame_path",
    "face_crop_path",
}

missing_video_cols = sorted(required_video_cols - set(df_videos_old.columns))
missing_face_cols = sorted(required_face_cols - set(df_final_faces.columns))

if missing_video_cols:
    raise ValueError(f"video_inventory.csv is missing columns: {missing_video_cols}")

if missing_face_cols:
    raise ValueError(f"final_identity_observations.csv is missing columns: {missing_face_cols}")


# Preserve the old video ordering exactly, but give the schema integer video IDs.
video_order = (
    df_videos_old[["video_id", "source_video"]]
    .drop_duplicates()
    .sort_values("video_id")
    .reset_index(drop=True)
)

video_id_map = {
    str(old_id): int(i)
    for i, old_id in enumerate(video_order["video_id"], start=1)
}

df_videos = df_videos_old.copy()
df_videos["old_video_id"] = df_videos["video_id"].astype(str)
df_videos["video_id"] = df_videos["old_video_id"].map(video_id_map)

if df_videos["video_id"].isna().any():
    raise ValueError("Could not map every video to a schema integer video_id.")

df_videos["video_id"] = df_videos["video_id"].astype(int)
df_videos["title"] = df_videos["source_video"].map(lambda x: Path(str(x)).stem)
df_videos["filename"] = df_videos["source_video"].astype(str)
df_videos["file_path"] = df_videos["source_path"].astype(str)
df_videos["fps"] = df_videos["reported_fps"].astype(float)

def file_created_at(path_text):
    path = Path(str(path_text))
    if not path.exists():
        raise FileNotFoundError(
            f"Original video is missing, so created_at cannot be derived: {path}"
        )
    stat = path.stat()
    ts = getattr(stat, "st_birthtime", stat.st_mtime)
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()

df_videos["created_at"] = df_videos["file_path"].map(file_created_at)


# Remap only the database video key. Person_ID assignments are unchanged.
df_final_faces["old_video_id"] = df_final_faces["video_id"].astype(str)
df_final_faces["video_id"] = df_final_faces["old_video_id"].map(video_id_map)

if df_final_faces["video_id"].isna().any():
    bad = df_final_faces.loc[
        df_final_faces["video_id"].isna(), "old_video_id"
    ].drop_duplicates().tolist()
    raise ValueError(f"Unmapped video IDs in final observations: {bad}")

df_final_faces["video_id"] = df_final_faces["video_id"].astype(int)
df_final_faces["source_frame_index"] = df_final_faces["source_frame_index"].astype(int)

# Rebuild frame_id to match the schema-ready integer video IDs.
df_final_faces["frame_id"] = df_final_faces.apply(
    lambda r: f"V{int(r['video_id']):04d}_F{int(r['source_frame_index']):09d}",
    axis=1,
)

# Deterministic FACE_DETECTION primary key.
df_final_faces = (
    df_final_faces
    .sort_values(
        [
            "video_id",
            "source_frame_index",
            "face_bbox_x1",
            "face_bbox_y1",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)

df_final_faces["detection_id"] = np.arange(
    1,
    len(df_final_faces) + 1,
    dtype=np.int64,
)

# Safety: no identity labels changed.
before_people = set(pd.read_csv(FINAL_FACE_OBSERVATIONS_CSV)["person_id"].astype(str))
after_people = set(df_final_faces["person_id"].astype(str))
assert before_people == after_people

print("Schema keys created.")
print("Person_IDs preserved:", len(after_people))
print("Detection IDs created:", len(df_final_faces))
display(
    df_final_faces[
        ["detection_id", "person_id", "video_id", "frame_id", "source_frame_index"]
    ].head(10)
)


In [ ]:
# ============================================================
# WRITE THE SIX DATABASE TABLES
# ============================================================

video_database = (
    df_videos[
        [
            "video_id",
            "title",
            "filename",
            "file_path",
            "created_at",
            "duration_seconds",
            "fps",
            "width",
            "height",
        ]
    ]
    .copy()
    .sort_values("video_id")
    .reset_index(drop=True)
)

frame_database = (
    df_final_faces[
        [
            "frame_id",
            "video_id",
            "source_frame_index",
            "source_time_seconds",
        ]
    ]
    .drop_duplicates("frame_id")
    .rename(columns={"source_frame_index": "frame_number"})
    .sort_values(["video_id", "frame_number"])
    .reset_index(drop=True)
)

frame_database["video_id"] = frame_database["video_id"].astype(int)
frame_database["frame_number"] = frame_database["frame_number"].astype(int)
frame_database["source_time_seconds"] = frame_database["source_time_seconds"].astype(float)

person_database = (
    df_final_faces[["person_id"]]
    .drop_duplicates()
    .sort_values("person_id")
    .reset_index(drop=True)
)

face_detection_database = (
    df_final_faces[
        [
            "detection_id",
            "person_id",
            "frame_id",
            "face_bbox_x1",
            "face_bbox_y1",
            "face_bbox_x2",
            "face_bbox_y2",
            "det_score",
        ]
    ]
    .rename(
        columns={
            "face_bbox_x1": "x1",
            "face_bbox_y1": "y1",
            "face_bbox_x2": "x2",
            "face_bbox_y2": "y2",
        }
    )
    .sort_values("detection_id")
    .reset_index(drop=True)
)

face_detection_database["detection_id"] = (
    face_detection_database["detection_id"].astype(int)
)

for col in ["x1", "y1", "x2", "y2", "det_score"]:
    face_detection_database[col] = face_detection_database[col].astype(float)


activity_database = pd.DataFrame({
    "activity_id": pd.Series(dtype="int64"),
    "detection_id": pd.Series(dtype="int64"),
    "start_frame": pd.Series(dtype="int64"),
    "end_frame": pd.Series(dtype="int64"),
    "activity_label": pd.Series(dtype="string"),
    "confidence": pd.Series(dtype="float64"),
})

engagement_database = pd.DataFrame({
    "engagement_id": pd.Series(dtype="int64"),
    "detection_id": pd.Series(dtype="int64"),
    "positive_score": pd.Series(dtype="float64"),
    "neutral_score": pd.Series(dtype="float64"),
    "negative_score": pd.Series(dtype="float64"),
})


VIDEO_CSV = DATABASE_DIR / "VIDEO.csv"
FRAME_CSV = DATABASE_DIR / "FRAME.csv"
PERSON_CSV = DATABASE_DIR / "PERSON.csv"
FACE_DETECTION_CSV = DATABASE_DIR / "FACE_DETECTION.csv"
ACTIVITY_CSV = DATABASE_DIR / "ACTIVITY.csv"
ENGAGEMENT_CSV = DATABASE_DIR / "ENGAGEMENT.csv"

video_database.to_csv(VIDEO_CSV, index=False)
frame_database.to_csv(FRAME_CSV, index=False)
person_database.to_csv(PERSON_CSV, index=False)
face_detection_database.to_csv(FACE_DETECTION_CSV, index=False)
activity_database.to_csv(ACTIVITY_CSV, index=False)
engagement_database.to_csv(ENGAGEMENT_CSV, index=False)

# Also save the enriched observations so detection_id is permanently retained.
SCHEMA_FACE_OBSERVATIONS_CSV = RESULTS_DIR / "final_identity_observations_schema_ready.csv"
df_final_faces.to_csv(SCHEMA_FACE_OBSERVATIONS_CSV, index=False)

print("Database tables written:")
for p in [
    VIDEO_CSV,
    FRAME_CSV,
    PERSON_CSV,
    FACE_DETECTION_CSV,
    ACTIVITY_CSV,
    ENGAGEMENT_CSV,
]:
    print("-", p)


In [ ]:
# ============================================================
# CREATE NEW RJ + MARION HANDOFFS USING detection_id
# ============================================================

downstream_faces = (
    df_final_faces[
        df_final_faces["selected_downstream_frame"].astype(bool)
    ]
    .copy()
)

# One Person_ID at most once per selected frame.
downstream_faces = (
    downstream_faces
    .sort_values(["face_quality_score", "det_score"], ascending=False)
    .drop_duplicates(["frame_id", "person_id"], keep="first")
)

downstream_faces["frame_number"] = (
    downstream_faces["source_frame_index"].astype(int)
)

downstream_faces["image_path"] = (
    downstream_faces["face_crop_path"]
    .map(lambda x: "FaceCrops/" + Path(str(x)).name)
)


# RJ
rj_engagement_input = (
    downstream_faces[
        [
            "detection_id",
            "person_id",
            "frame_id",
            "video_id",
            "frame_number",
            "source_time_seconds",
            "image_path",
            "det_score",
        ]
    ]
    .sort_values(["video_id", "frame_number", "detection_id"])
    .reset_index(drop=True)
)

RJ_HANDOFF_CSV = HANDOFF_DIR / "rj_engagement_input.csv"
rj_engagement_input.to_csv(RJ_HANDOFF_CSV, index=False)


# Marion person/frame map
marion_activity_input = (
    downstream_faces[
        [
            "detection_id",
            "person_id",
            "frame_id",
            "video_id",
            "frame_number",
            "source_time_seconds",
            "full_frame_path",
            "face_crop_path",
            "face_bbox_x1",
            "face_bbox_y1",
            "face_bbox_x2",
            "face_bbox_y2",
            "det_score",
        ]
    ]
    .rename(
        columns={
            "face_bbox_x1": "x1",
            "face_bbox_y1": "y1",
            "face_bbox_x2": "x2",
            "face_bbox_y2": "y2",
        }
    )
    .sort_values(["video_id", "frame_number", "detection_id"])
    .reset_index(drop=True)
)

MARION_HANDOFF_CSV = HANDOFF_DIR / "marion_activity_input.csv"
marion_activity_input.to_csv(MARION_HANDOFF_CSV, index=False)


# Marion unique frames: run the activity model once per selected full frame.
activity_rows = []

for frame_id, group in marion_activity_input.groupby("frame_id", sort=False):
    first = group.iloc[0]

    people = [
        {
            "detection_id": int(row["detection_id"]),
            "person_id": str(row["person_id"]),
            "face_bbox": {
                "x1": int(float(row["x1"])),
                "y1": int(float(row["y1"])),
                "x2": int(float(row["x2"])),
                "y2": int(float(row["y2"])),
            },
        }
        for _, row in group.iterrows()
    ]

    activity_rows.append({
        "frame_id": str(frame_id),
        "video_id": int(first["video_id"]),
        "frame_number": int(first["frame_number"]),
        "source_time_seconds": float(first["source_time_seconds"]),
        "full_frame_path": str(first["full_frame_path"]),
        "person_count": int(len(group)),
        "persons_json": json.dumps(people),
    })

activity_frames_unique = (
    pd.DataFrame(activity_rows)
    .sort_values(["video_id", "frame_number"])
    .reset_index(drop=True)
)

ACTIVITY_UNIQUE_CSV = HANDOFF_DIR / "activity_frames_unique.csv"
activity_frames_unique.to_csv(ACTIVITY_UNIQUE_CSV, index=False)

print("RJ rows:", len(rj_engagement_input))
print("Marion person/frame rows:", len(marion_activity_input))
print("Marion unique frames:", len(activity_frames_unique))
print()
print("RJ:", RJ_HANDOFF_CSV)
print("Marion:", MARION_HANDOFF_CSV)
print("Unique frames:", ACTIVITY_UNIQUE_CSV)


In [ ]:
# ============================================================
# VALIDATE SCHEMA + FOREIGN KEYS
# ============================================================

VIDEO_COLUMNS = [
    "video_id",
    "title",
    "filename",
    "file_path",
    "created_at",
    "duration_seconds",
    "fps",
    "width",
    "height",
]

FRAME_COLUMNS = [
    "frame_id",
    "video_id",
    "frame_number",
    "source_time_seconds",
]

PERSON_COLUMNS = ["person_id"]

FACE_DETECTION_COLUMNS = [
    "detection_id",
    "person_id",
    "frame_id",
    "x1",
    "y1",
    "x2",
    "y2",
    "det_score",
]

ACTIVITY_COLUMNS = [
    "activity_id",
    "detection_id",
    "start_frame",
    "end_frame",
    "activity_label",
    "confidence",
]

ENGAGEMENT_COLUMNS = [
    "engagement_id",
    "detection_id",
    "positive_score",
    "neutral_score",
    "negative_score",
]

face_detection_ids = set(
    face_detection_database["detection_id"].astype(int)
)

validation = {
    "video_schema_exact": list(video_database.columns) == VIDEO_COLUMNS,
    "frame_schema_exact": list(frame_database.columns) == FRAME_COLUMNS,
    "person_schema_exact": list(person_database.columns) == PERSON_COLUMNS,
    "face_detection_schema_exact": (
        list(face_detection_database.columns) == FACE_DETECTION_COLUMNS
    ),
    "activity_schema_exact": list(activity_database.columns) == ACTIVITY_COLUMNS,
    "engagement_schema_exact": list(engagement_database.columns) == ENGAGEMENT_COLUMNS,

    "duplicate_video_id": int(video_database["video_id"].duplicated().sum()),
    "duplicate_video_filename": int(video_database["filename"].duplicated().sum()),
    "duplicate_frame_id": int(frame_database["frame_id"].duplicated().sum()),
    "duplicate_person_id": int(person_database["person_id"].duplicated().sum()),
    "duplicate_detection_id": int(
        face_detection_database["detection_id"].duplicated().sum()
    ),

    "bad_frame_video_fk": int(
        (~frame_database["video_id"].isin(video_database["video_id"])).sum()
    ),
    "bad_detection_person_fk": int(
        (~face_detection_database["person_id"].isin(person_database["person_id"])).sum()
    ),
    "bad_detection_frame_fk": int(
        (~face_detection_database["frame_id"].isin(frame_database["frame_id"])).sum()
    ),
    "duplicate_person_per_frame": int(
        face_detection_database
        .duplicated(["frame_id", "person_id"])
        .sum()
    ),
    "bad_rj_detection_id": int(
        (~rj_engagement_input["detection_id"].astype(int).isin(face_detection_ids)).sum()
    ),
    "bad_marion_detection_id": int(
        (~marion_activity_input["detection_id"].astype(int).isin(face_detection_ids)).sum()
    ),
    "duplicate_activity_frames": int(
        activity_frames_unique["frame_id"].duplicated().sum()
    ),
}

for key in [
    "video_schema_exact",
    "frame_schema_exact",
    "person_schema_exact",
    "face_detection_schema_exact",
    "activity_schema_exact",
    "engagement_schema_exact",
]:
    assert validation[key], key

for key in [
    "duplicate_video_id",
    "duplicate_video_filename",
    "duplicate_frame_id",
    "duplicate_person_id",
    "duplicate_detection_id",
    "bad_frame_video_fk",
    "bad_detection_person_fk",
    "bad_detection_frame_fk",
    "duplicate_person_per_frame",
    "bad_rj_detection_id",
    "bad_marion_detection_id",
    "duplicate_activity_frames",
]:
    assert validation[key] == 0, f"{key}: {validation[key]}"

print(json.dumps(validation, indent=2))
print()
print("SCHEMA PREP VALIDATION: PASSED")
print()
print("No identity models or clustering were rerun.")


## Result

After the validation cell passes, use the files under:

```text
Data/IdentityPipelineTop2/<RUN_MODE>/Database/
├── VIDEO.csv
├── FRAME.csv
├── PERSON.csv
├── FACE_DETECTION.csv
├── ACTIVITY.csv
└── ENGAGEMENT.csv
```

and the updated handoffs under:

```text
Data/IdentityPipelineTop2/<RUN_MODE>/Handoffs/
├── rj_engagement_input.csv
├── marion_activity_input.csv
└── activity_frames_unique.csv
```

Your original `Person_ID` clustering remains unchanged.
